In [4]:
import pandas as pd

df = pd.read_csv("Datasets/kaggle 2015 flight delays/flights.csv", low_memory=False)

df = df[df["AIRLINE"] == "AA"]

df = df[df["CANCELLED"] == 0]

df = df[[
    "YEAR", "MONTH", "DAY",
    "TAIL_NUMBER",
    "AIRLINE",
    "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
    "SCHEDULED_DEPARTURE", "DEPARTURE_TIME",
    "SCHEDULED_ARRIVAL", "ARRIVAL_TIME",
    "DEPARTURE_DELAY", "ARRIVAL_DELAY"
]]

df = df.dropna()

df["FL_DATE"] = pd.to_datetime(df[["YEAR", "MONTH", "DAY"]])

df.to_csv("flights_clean.csv", index=False)
print("Saved flights_clean.csv with", len(df), "rows")


Saved flights_clean.csv with 712935 rows


In [5]:
from datetime import timedelta

df = pd.read_csv("flights_clean.csv", parse_dates=["FL_DATE"])

def time_to_datetime(row, col_name):
    """
    يحوّل وقت بالشكل HHMM (مثلاً 1423) لـ datetime كامل مع اليوم.
    """
    t = str(int(row[col_name])).zfill(4)  # "930" -> "0930"
    hour = int(t[:2])
    minute = int(t[2:])
    return row["FL_DATE"] + timedelta(hours=hour, minutes=minute)

df["arr"] = df.apply(lambda r: time_to_datetime(r, "ARRIVAL_TIME"), axis=1)
df["dep"] = df.apply(lambda r: time_to_datetime(r, "DEPARTURE_TIME"), axis=1)

events = []

for _, r in df.iterrows():
    case_id = f"{r['TAIL_NUMBER']}_{r['FL_DATE'].date()}"

    arr = r["arr"]
    dep = r["dep"]
    tat = dep - arr

    if tat <= timedelta(minutes=15):
        continue

    stages = {
        "Gate arrival": 0.00,
        "Deboarding start": 0.08,
        "Cleaning start": 0.25,
        "Fueling start": 0.30,
        "Boarding start": 0.65,
        "Boarding end": 0.90,
        "Pushback": 1.00
    }

    for activity, pct in stages.items():
        t = arr + pct * tat
        events.append({
            "case:concept:name": case_id,
            "concept:name": activity,
            "time:timestamp": t,
            "origin": r["ORIGIN_AIRPORT"],
            "destination": r["DESTINATION_AIRPORT"],
            "dep_delay": r["DEPARTURE_DELAY"],
            "arr_delay": r["ARRIVAL_DELAY"],
            "airline": r["AIRLINE"],
            "tail_number": r["TAIL_NUMBER"]
        })

events_df = pd.DataFrame(events)
events_df = events_df.sort_values(["case:concept:name", "time:timestamp"])

events_df.to_csv("turnaround_log.csv", index=False)

print("Saved turnaround_log.csv with", events_df["case:concept:name"].nunique(), "turnarounds")
print("Total events:", len(events_df))


Saved turnaround_log.csv with 30550 turnarounds
Total events: 213927
